# Gradient-Variance (Barren-Plateau) Analysis

Investigates the training diagnostic: the mean parameter-gradient variance
over samples within a batch, recorded per gradient step. It is reported as
a trajectory per approach, without a binary detection threshold.

In [ ]:
import os
import sys
from pathlib import Path

# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


## 1. Load Training Diagnostics

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

APPROACHES = ["baseline", "layerwise", "local_cost"]
DEPTHS = [4, 6, 8]

trajectories = {}   # (approach, depth) -> list of (step, value) per seed
for approach in APPROACHES:
    for depth in DEPTHS:
        base = ROOT / "results" / approach / f"depth_{depth}"
        runs = []
        for seed_dir in sorted(base.glob("seed_*")):
            mf = seed_dir / "metrics.json"
            if not mf.exists():
                continue
            m = json.loads(mf.read_text())
            traj = m.get("training_diagnostic", {}).get("trajectory", {})
            if traj.get("step"):
                runs.append((traj["step"], traj["mean_param_grad_variance"]))
        trajectories[(approach, depth)] = runs
print(f"Loaded diagnostics for "
      f"{sum(bool(v) for v in trajectories.values())} of "
      f"{len(trajectories)} (approach, depth) combinations.")


## 2. Gradient-Variance Trajectories

Mean over seeds (shaded = mean +/- SD),
one panel per depth.

In [ ]:
fig, axes = plt.subplots(1, len(DEPTHS), figsize=(18, 5), squeeze=False)
for col, depth in enumerate(DEPTHS):
    ax = axes[0, col]
    for approach in APPROACHES:
        runs = trajectories.get((approach, depth), [])
        if not runs:
            continue
        steps = runs[0][0]
        grid = np.vstack([
            np.interp(steps, np.asarray(s), np.asarray(v)) for s, v in runs
        ])
        mean = grid.mean(axis=0)
        sd = grid.std(axis=0)
        ax.plot(steps, mean, label=approach, lw=2)
        ax.fill_between(steps, mean - sd, mean + sd, alpha=0.2)
    ax.set_yscale("log")
    ax.set_title(f"depth {depth}")
    ax.set_xlabel("Gradient step")
    ax.set_ylabel("Mean param-gradient variance")
    ax.legend(fontsize="small")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Cross-Approach Summary Table

In [ ]:
print(f"{'approach':11s} {'depth':>5} {'mean':>12} {'std':>12} {'n':>3}")
for approach in APPROACHES:
    for depth in DEPTHS:
        vals = [v for _, vs in trajectories.get((approach, depth), [])
                for v in vs]
        if vals:
            print(f"{approach:11s} {depth:>5} {np.mean(vals):>12.3e} "
                  f"{np.std(vals):>12.3e} {len(vals):>3}")


## 4. Landscape Analysis

Loads `variance_scaling.json` (if produced by the
circuit-landscape study) and plots the landscape statistic vs qubit count.

In [ ]:
from src.evaluation import plot_variance_scaling

vs_file = ROOT / "results" / "variance_scaling" / "variance_scaling.json"
if vs_file.exists():
    data = json.loads(vs_file.read_text())
    print(f"Landscape configurations: {len(data['configs'])}")
    plot_variance_scaling(data["configs"], ROOT / "variance_vs_n.png", x_axis="n")
else:
    print("No variance_scaling.json found. Run run_variance_scaling.py first.")


## 5. Conclusions

- How the gradient variance evolves during training per
  approach and depth.
- Whether global and local cost differ in their
  gradient-variance trajectories.